# 1. Fase de preparación

En este documento se omite la fase de obtención y filtrado de los datos, comenzando directamente desde la integración y la construcción de las muestras. Se preparas y unifican los datos para entrenar un modelo predictivo de accidentes de tráfico a través de tres pasos clave:

- Integración de datos: Cruza los registros de accidentes con información del clima, los datos de la red vial y el calendario.

- Generación de muestra de entrenamiento: Mezcla accidentes reales (2019-2023) con registros de control simulados donde no hubo choques, para enseñarle al modelo a identificar los factores de riesgo.

- Preparación de muestras de prueba: Reserva el año 2024 y diseña escenarios específicos (por ejemplo, días lluviosos frente a soleados) para medir la precisión de las predicciones del modelo.

## 1. Configuración y librerías

In [1]:
import sys
import subprocess

# Diccionario con el nombre del módulo al importar y el nombre del paquete en pip
librerias_requeridas = {
    'pandas': 'pandas',
    'geopandas': 'geopandas',
    'numpy': 'numpy',
    'swifter': 'swifter'
}

# Verificación e instalación dinámica
for modulo, paquete in librerias_requeridas.items():
    try:
        __import__(modulo)
    except ImportError:
        print(f"La librería '{modulo}' no se encontró. Iniciando instalación...")
        # Ejecuta el comando pip install usando el ejecutable actual de Python
        subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])
        print(f"{modulo} se instaló exitosamente.")

# Una vez garantizada la existencia de las librerías, hacemos los imports estándar
import pandas as pd
import geopandas as gpd
import numpy as np
import random
import swifter
import os

# Configuración original de tu notebook
pd.set_option('display.max_columns', None)

print("Configuración inicial completada. Todo listo para trabajar.")

Configuración inicial completada. Todo listo para trabajar.


## 2. Lectura de datos

In [2]:
# Cargar los datos de accidentes
df_acc = pd.read_parquet("Generados/accidentes_franjas.parquet")
print("Datos de accidentes cargados")
display(df_acc.head(4))
print(df_acc.info())

Datos de accidentes cargados


,ID_ACC,ID_RED,FECHA_FRANJA,DIASEMANA
0,0,663941,2019-01-03 00:00:00,3
1,1,549969_2,2019-01-03 06:00:00,3
2,2,4454776,2019-01-03 06:00:00,3
3,3,2617146,2019-01-03 12:00:00,3


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19168 entries, 0 to 19167
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   ID_ACC        19168 non-null  Int16         
 1   ID_RED        19168 non-null  object        
 2   FECHA_FRANJA  19168 non-null  datetime64[ns]
 3   DIASEMANA     19168 non-null  Int8          
dtypes: Int16(1), Int8(1), datetime64[ns](1), object(1)
memory usage: 393.2+ KB
None


In [3]:
# Cargar datos meteorológicos
df_meteo = pd.read_parquet("Generados/consultas_meteo_franjas.parquet")
print("Datos meteorológicos cargados")
display(df_meteo.head(4))
print(df_meteo.info())

Datos meteorológicos cargados


,ID_CENTRO,FECHA_FRANJA,PRECIP_SUM,PRECIP_MAX,WIND_GUSTS_MAX,WIND_SPEED_MEAN,TEMP_MEAN,TEMP_MIN,APP_TEMP_MEAN,CLOUD_COVER_MEAN,SOIL_TEMP_MEAN,SOIL_MOISTURE_MEAN
0,0,2019-01-01 00:00:00,0.0,0.0,11.880000,3.484983,8.425000,7.70,6.281617,31.833334,9.308333,0.112000
1,0,2019-01-01 06:00:00,0.0,0.0,16.200001,4.029133,10.925000,7.05,9.084033,59.833332,9.358334,0.113667
2,0,2019-01-01 12:00:00,0.0,0.0,32.040001,9.749784,20.183332,18.25,17.700583,72.166664,21.333334,0.108333
3,0,2019-01-01 18:00:00,0.0,0.0,26.280001,4.038717,15.891666,13.50,14.537150,99.166664,16.708334,0.105000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5146816 entries, 0 to 5146815
Data columns (total 12 columns):
 #   Column              Dtype         
---  ------              -----         
 0   ID_CENTRO           int16         
 1   FECHA_FRANJA        datetime64[ns]
 2   PRECIP_SUM          float32       
 3   PRECIP_MAX          float32       
 4   WIND_GUSTS_MAX      float32       
 5   WIND_SPEED_MEAN     float32       
 6   TEMP_MEAN           float32       
 7   TEMP_MIN            float32       
 8   APP_TEMP_MEAN       float32       
 9   CLOUD_COVER_MEAN    float32       
 10  SOIL_TEMP_MEAN      float32       
 11  SOIL_MOISTURE_MEAN  float32       
dtypes: datetime64[ns](1), float32(10), int16(1)
memory usage: 245.4 MB
None


In [4]:
# Cargar los datos de la red vial
gdf_rnc = gpd.read_parquet("Generados/rnc_ags.parquet")
print("Datos de la red vial cargados")
display(gdf_rnc.head(4))
print(gdf_rnc.info())

Datos de la red vial cargados


,ID_RED,ID_CENTRO,CARRILES,NIVEL,VELOCIDAD,LONGITUD,UNION_INI,UNION_FIN,AVENIDA,BOULEVARD,CALLE,CAMINO,CARRETERA,CIRCUITO,GLORIETA,INCORPORACION,PRIVADA,RETORNO_U,MENOR,CERRADA_EN_AMBOS_SENTIDOS,DOS_SENTIDOS,UN_SENTIDO,GRADO_MIN,GRADO_MAX,DIST_NODO_3,DIST_NODO_4,DIST_NODO_5_MAS,SINUOSIDAD,DIST_MAN_PROH,DELTA_CARRILES_MAX,DELTA_CARRILES_MIN,DELTA_CARRILES_MEAN,DELTA_VELOCIDAD_MAX,DELTA_VELOCIDAD_MIN,DELTA_VELOCIDAD_MEAN,geometry
0,358174,322,2,0,80,19.447878,1172947,1172948,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,4,9.780806,9.780806,9575.637695,1.00007,0.000000,1,0,0.666667,50,20,40.000000,"LINESTRING (-102.31006 22.45709, -102.31007 22..."
1,358196_1,583,2,0,60,141.066162,1172976,3342859,0,0,0,0,1,0,0,0,0,0,0,0,1,0,2,2,211.588486,10855.881836,16497.392578,1.00007,9287.947266,0,0,0.000000,0,0,0.000000,"LINESTRING (-101.87234 22.03214, -101.87133 22..."
2,358196_2,583,2,0,60,141.066162,3342859,1172977,0,0,0,0,1,0,0,0,0,0,0,0,1,0,2,3,70.533081,10995.713867,16635.238281,1.00000,9426.706055,0,0,0.000000,20,0,6.666667,"LINESTRING (-101.87104 22.03173, -101.87101 22..."
3,358199_1,448,2,0,60,195.425568,1172981,3342860,0,0,0,0,1,0,0,0,0,0,0,0,1,0,2,2,404.616425,7020.501953,12812.265625,1.00027,7020.501953,0,0,0.000000,20,0,10.000000,"LINESTRING (-102.16663 22.31939, -102.16635 22..."


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 63010 entries, 0 to 63009
Data columns (total 36 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   ID_RED                     63010 non-null  object  
 1   ID_CENTRO                  63010 non-null  int16   
 2   CARRILES                   63010 non-null  int8    
 3   NIVEL                      63010 non-null  int8    
 4   VELOCIDAD                  63010 non-null  int16   
 5   LONGITUD                   63010 non-null  float32 
 6   UNION_INI                  63010 non-null  int32   
 7   UNION_FIN                  63010 non-null  int32   
 8   AVENIDA                    63010 non-null  int8    
 9   BOULEVARD                  63010 non-null  int8    
 10  CALLE                      63010 non-null  int8    
 11  CAMINO                     63010 non-null  int8    
 12  CARRETERA                  63010 non-null  int8    
 13  CIRCUITO               

In [5]:
# Cargar el calendario no ordinario
df_cal = pd.read_parquet("Generados/calendario.parquet")
print("Calendario no ordinario cargado")
display(df_cal.head(4))
print(df_cal.info())

Calendario no ordinario cargado


,FECHA,FESTIVO,FNSM,VACACIONES,PROMOCION,PANDEMIA
0,2019-01-01,1,0,1,0,0
1,2019-01-02,0,0,1,0,0
2,2019-01-03,0,0,1,0,0
3,2019-01-04,0,0,1,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2192 entries, 0 to 2191
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   FECHA       2192 non-null   datetime64[ns]
 1   FESTIVO     2192 non-null   int8          
 2   FNSM        2192 non-null   int8          
 3   VACACIONES  2192 non-null   int8          
 4   PROMOCION   2192 non-null   int8          
 5   PANDEMIA    2192 non-null   int8          
dtypes: datetime64[ns](1), int8(5)
memory usage: 28.0 KB
None


## 4. Separación de conjuntos

El conjunto de registros de accidentes se separó de la siguiente manera: 2019 – 2023 para la generación de la muestra de entrenamiento; y 2024 para la muestra de prueba.

In [6]:
fecha_corte = '2024-01-01'
df_acc_train = df_acc[df_acc['FECHA_FRANJA'] < fecha_corte]
df_acc_test = df_acc[df_acc['FECHA_FRANJA'] >= fecha_corte]

# Resumen
df_resumen_acc = pd.DataFrame([
    {
        "Conjunto": "Entrenamiento",
        "Fecha_inicio": df_acc_train["FECHA_FRANJA"].min().date(),
        "Fecha_final": df_acc_train["FECHA_FRANJA"].max().date(),
        "Conteo": len(df_acc_train),
        "Porcentaje": round(len(df_acc_train) / len(df_acc) * 100,2)
    },
    {
        "Conjunto": "Prueba",
        "Fecha_inicio": df_acc_test["FECHA_FRANJA"].min().date(),
        "Fecha_final": df_acc_test["FECHA_FRANJA"].max().date(),
        "Conteo": len(df_acc_test),
        "Porcentaje": round(len(df_acc_test) / len(df_acc) * 100,2)
    }
])

display(df_resumen_acc)


,Conjunto,Fecha_inicio,Fecha_final,Conteo,Porcentaje
0,Entrenamiento,2019-01-01,2023-12-31,15714,81.98
1,Prueba,2024-01-01,2024-12-31,3454,18.02


## 5. Muestra de entrenamiento

### 5.1 Generación de controles

Con la finalidad de que el modelo predictivo aprenda a distinguir las diferentes condiciones que son propicias para que ocurran los accidentes viales, es necesario mostrarle escenarios en donde no ocurren accidentes.

Se generó una muestra de control en relación 1:4 respecto a estos registros para mejorar la precisión del modelo predictivo. Por cada registro, se generaron dos controles espaciales y dos controles temporales con diseño de caso-cruzado:
* Los controles espaciales asignan aleatoriamente otros segmentos de la red para las mismas franjas temporales. 
* Los controles temporales mantienen fijo el segmento y realizan una regresión hacia atrás en el tiempo (una y dos semanas para este caso). 

Posteriormente, se realiza un filtrado anti-solapamiento, eliminando aquellos casos de control que coincidan en lugar y franja con algún registro de los accidentes.

In [7]:
def generar_muestra_hibrida_franjas(df_accidentes: pd.DataFrame, ids_red_vial: pd.Series, semanas_control:list = [1, 2], random_state=42) -> pd.DataFrame:
    """
    Genera una muestra híbrida de casos y controles para análisis de accidentes
    en franjas horarias de 6 horas, combinando controles temporales y espaciales.

    Parámetros
    ----------
    df_accidentes : DataFrame con registros de accidentes. 
        Debe contener al menos:
        - 'ID_RED' : identificador de la calle/tramo vial
        - 'FECHA_FRANJA' : fecha y franja horaria (tipo datetime)

    ids_red_vial : Lista o arreglo con todos los identificadores de la red vial
        (se utiliza para asignar controles espaciales aleatorios).

    semanas_control : Lista o arreglo con el número de semanas a desplazar para generar controles temporales (default=[1, 2]).
        Cada valor genera un conjunto de controles (ej. 1 semana, 2 semanas).

    random_state : Opcional (default=42)
        Semilla para reproducibilidad en la selección aleatoria de controles espaciales.

    Retorna
    -------
    df_final : DataFrame con casos y controles, 
        con las siguientes columnas:
        - 'ID_RED' : identificador de tramo vial
        - 'FECHA_FRANJA' : fecha y franja horaria asociada
        - 'TARGET' : 1 para accidente real, 0 para control
        - 'TIPO_MUESTRA' : tipo de muestra ('accidente_real', 'ctrl_temp_Xw', 'ctrl_esp_Y')
        - 'DIASEMANA' : día de la semana (0=lunes,..., 6=domingo)
    """
    np.random.seed(random_state)
    random.seed(random_state)
    
    # ---------------------------------------------------------
    # CASOS POSITIVOS
    # ---------------------------------------------------------
    df_casos = df_accidentes.copy()
    
    df_casos['TARGET'] = 1
    df_casos['TIPO_MUESTRA'] = 'accidente_real'

    # Obtener límites temporales de las franjas
    fecha_minima = df_casos['FECHA_FRANJA'].min()
    fecha_maxima = df_casos['FECHA_FRANJA'].max()
    
    lista_controles = []
    
    # ---------------------------------------------------------
    # CONTROLES TEMPORALES (1 y 2 semanas)
    # ---------------------------------------------------------
    for semana in semanas_control:
        df_temp = df_casos[['ID_RED', 'FECHA_FRANJA']].copy()
        fecha_original = df_temp['FECHA_FRANJA'].copy()
        
        # Intento por defecto: hacia el pasado
        fecha_pasado = fecha_original - pd.to_timedelta(semana, unit='W')
        mask_fuera_inferior = fecha_pasado < fecha_minima
        
        # Asignar fechas (pasado por defecto, futuro si se sale por abajo)
        df_temp['FECHA_FRANJA'] = fecha_pasado
        df_temp.loc[mask_fuera_inferior, 'FECHA_FRANJA'] = fecha_original[mask_fuera_inferior] + pd.to_timedelta(semana, unit='W')
        
        # Descartar si la corrección hacia el futuro nos sacó del máximo
        df_temp = df_temp[df_temp['FECHA_FRANJA'] <= fecha_maxima]
        
        df_temp['TARGET'] = 0
        df_temp['TIPO_MUESTRA'] = f'ctrl_temp_{semana}w'
            
        lista_controles.append(df_temp)

    # ---------------------------------------------------------
    # CONTROLES ESPACIALES (2 individuos)
    # ---------------------------------------------------------
    for i in range(1, 3):
        df_esp = df_casos[['ID_RED', 'FECHA_FRANJA']].copy()
        
        # Asignar una calle aleatoria de TODA la red para cada registro de franja
        df_esp['ID_RED'] = np.random.choice(ids_red_vial, size=len(df_casos))
        
        df_esp['TARGET'] = 0
        df_esp['TIPO_MUESTRA'] = f'ctrl_esp_{i}'
            
        lista_controles.append(df_esp)

    # ---------------------------------------------------------
    # FILTRADO ANTI-SOLAPAMIENTO
    # ---------------------------------------------------------
    # Unimos todos los controles en un solo DataFrame temporal
    df_todos_controles = pd.concat(lista_controles, ignore_index=True)
    
    # Creamos un registro de los eventos reales basado en FRANJA y CALLE
    casos_unicos = df_casos[['ID_RED', 'FECHA_FRANJA']].drop_duplicates()
    
    # Left Merge para identificar controles que simularon una calle y franja donde si hubo choque real
    df_todos_controles = df_todos_controles.merge(
        casos_unicos,
        on=['ID_RED', 'FECHA_FRANJA'],
        how='left',
        indicator=True
    )
    
    # Nos quedamos solamente  con los que no hicieron match
    df_controles_limpios = df_todos_controles[df_todos_controles['_merge'] == 'left_only'].drop(columns=['_merge'])
    
    # ---------------------------------------------------------
    # UNIÓN CASOS Y CONTROLES
    # ---------------------------------------------------------
    # Seleccionamos y ordenamos las columnas clave
    cols_finales = ['ID_RED', 'FECHA_FRANJA', 'TARGET', 'TIPO_MUESTRA']
    
    df_final = pd.concat([df_casos[cols_finales], df_controles_limpios[cols_finales]], ignore_index=True)
    df_final['TARGET'] = df_final['TARGET'].astype('int8')
    df_final['DIASEMANA'] = df_final['FECHA_FRANJA'].dt.dayofweek.astype('int8')
    
    return df_final

# Uso:
df_muestra_train = generar_muestra_hibrida_franjas(df_acc_train, gdf_rnc['ID_RED'].unique())
df_muestra_train.head(5)

,ID_RED,FECHA_FRANJA,TARGET,TIPO_MUESTRA,DIASEMANA
0,663941,2019-01-03 00:00:00,1,accidente_real,3
1,549969_2,2019-01-03 06:00:00,1,accidente_real,3
2,4454776,2019-01-03 06:00:00,1,accidente_real,3
3,2617146,2019-01-03 12:00:00,1,accidente_real,3
4,555701,2019-01-03 18:00:00,1,accidente_real,3


### 5.2. Sacrificio de los primero tres meses para la memoria

Para que el modelo identifique posibles *hotspots*, se agregaron variables de memoria temporal, por lo que se utilizaron los primeros 56 días (8 semanas de 7 días) para obtener dichas variables y evitar arranques en frio.

In [8]:
# Encontrar la franja más antigua
fecha_inicio = df_muestra_train['FECHA_FRANJA'].min()

# Calcular la fecha de corte sumando exactamente 2 meses
fecha_corte = fecha_inicio + pd.Timedelta(days=56) # 2 meses = 56 días (asumiendo semanas de 7 días para conservar día de la semana)

print(f"La recolección de datos inicia el: {fecha_inicio.date()}")
print(f"Los registros válidos comenzarán a partir del: {fecha_corte}")

# Filtrado (solo lo que sea mayor o igual a la fecha de corte)
df_train = df_muestra_train[df_muestra_train['FECHA_FRANJA'] >= fecha_corte].copy()
df_train = df_train.reset_index(drop=True)

print(f"Total de registros originales: {len(df_muestra_train)}")
print(f"Total de registros después de quitar los primeros 2 meses: {len(df_train)}")
print(f"Se han eliminado {len(df_muestra_train) - len(df_train)}",
      f"({((len(df_muestra_train) - len(df_train))/len(df_muestra_train)*100):.2f}%)",
      "registros correspondientes a los primeros 2 meses de la muestra de entrenamiento.")
display(df_train)

La recolección de datos inicia el: 2019-01-01
Los registros válidos comenzarán a partir del: 2019-02-26 00:00:00
Total de registros originales: 78493
Total de registros después de quitar los primeros 2 meses: 76722
Se han eliminado 1771 (2.26%) registros correspondientes a los primeros 2 meses de la muestra de entrenamiento.


,ID_RED,FECHA_FRANJA,TARGET,TIPO_MUESTRA,DIASEMANA
0,877898_2,2019-02-28 06:00:00,1,accidente_real,3
1,550482,2019-02-28 06:00:00,1,accidente_real,3
2,550075_1,2019-02-28 00:00:00,1,accidente_real,3
3,4838974,2019-02-27 06:00:00,1,accidente_real,2
4,2639756,2019-02-28 00:00:00,1,accidente_real,3
...,...,...,...,...,...
76717,2399447_3,2023-12-08 06:00:00,0,ctrl_esp_2,4
76718,3537966,2023-12-10 12:00:00,0,ctrl_esp_2,6
76719,4118057,2023-12-20 00:00:00,0,ctrl_esp_2,2
76720,4121748,2023-12-22 06:00:00,0,ctrl_esp_2,4


## 6. Instancias de prueba

Dado que el objetivo es crear mapas de riesgo en diversas franjas de tiempo, el modelo predecirá sobre todas las aristas de la red vial de Aguascalientes, por lo que el factor dinámico serán la fecha y la franja de tiempo, con los caracteristicas meteorológicas y contextuales de la misma. Las instancias de prueba se seleccionaron de manera controlada para validar la sensibilidad y la coherencia del modelo respecto a estas variables dinámicas.

Para contrastar la climatología, se identificaron pares de instancias que compartían exactamente el mismo día de la semana y la misma franja, pero difieren drásticamente en su historial de precipitación y viento. Para el contraste de la rutina, se seleccionaron pares de instancias que coinciden en hora y tienen clima similar, pero representan condiciones sociales opuestas. Finalmente, para observar la evolución de franjas consecutivas, se extrajeron bloques completos y secuenciales de franjas horarias.

In [10]:
def generar_dataset_test(df_calendario: pd.DataFrame, df_meteo: pd.DataFrame, df_accidentes_test: pd.DataFrame) -> pd.DataFrame:
    """
    Genera un dataset experimental de prueba para 2024, integrando clima, calendario y accidentes, con selección controlada de instancias.

    Parámetros
    ----------
    df_fechas: Calendario de fechas no ordinarias con variables de contexto (ej. festivos, fines de semana, vacaciones).
        Debe contener al menos:
        - 'FECHA' : fecha normalizada (datetime)
        - 'FESTIVO', 'FNSM', etc.

    df_meteo : Datos meteorológicos históricos por franja horaria.
        Debe contener:
        - 'FECHA_FRANJA' : datetime de cada franja (6h)
        - Variables climáticas como 'TEMP_MEAN', 'PRECIP_SUM', etc.

    df_accidentes_test : Accidentes reales en 2024, 
        con columna:
        - 'FECHA_FRANJA' : datetime de la franja horaria del accidente.

    Retorna
    -------
    df_final : Dataset consolidado con 60 instancias experimentales, 
        incluyendo:
        - Contexto temporal (día, mes, franja, festivo, etc.).
        - Variables climáticas (medianas y lag de la franja anterior).
        - Conteo de accidentes reales en cada franja.
        - Etiquetas metodológicas e identificadores de franja.
    """
    
    # PROCESAMIENTO DE CLIMA
    # Obtener la mediana global de la red, para cada franja
    meteo_agrupada = df_meteo.groupby('FECHA_FRANJA').median(numeric_only=True).reset_index()
    meteo_agrupada = meteo_agrupada.sort_values('FECHA_FRANJA').drop(columns=["ID_CENTRO"])

    cols_clima_base = [
        'WIND_SPEED_MEAN', 'SOIL_MOISTURE_MEAN', 'APP_TEMP_MEAN', 
        'TEMP_MEAN', 'WIND_GUSTS_MAX', 'SOIL_TEMP_MEAN', 'TEMP_MIN', 
        'CLOUD_COVER_MEAN', 'WEATHER_MAX_CONT', 'PRECIP_SUM', 'PRECIP_MAX'
    ]
    
    # Lag (clima de la franja anterior)
    for col in cols_clima_base:
        if col in meteo_agrupada.columns:
            meteo_agrupada[f'{col}_PREV'] = meteo_agrupada[col].shift(1)

    # Filtrado 2024 y merge con calendario
    meteo_24 = meteo_agrupada[meteo_agrupada['FECHA_FRANJA'].dt.year == 2024].copy()
    fechas_24 = df_calendario[df_calendario['FECHA'].dt.year == 2024].copy()
    
    df = pd.merge(
        meteo_24, 
        fechas_24, 
        left_on=meteo_24['FECHA_FRANJA'].dt.normalize(), 
        right_on='FECHA', 
        how='inner'
    )
    
    df['DIA_SEM'] = df['FECHA_FRANJA'].dt.dayofweek
    df['HORA'] = df['FECHA_FRANJA'].dt.hour

    # -------------------------
    # SELECCIÓN EXPERIMENTAL
    # -------------------------
    rng = np.random.default_rng(seed=42)
    seleccionados = set()
    rows = []

    # Se añade el parámetro id_franja a la función de guardado
    def add_r(row, met, id_franja):
        if row['FECHA_FRANJA'] not in seleccionados:
            seleccionados.add(row['FECHA_FRANJA'])
            d = row.to_dict()
            d['METODOLOGIA'] = met
            d['ID_FRANJA'] = id_franja
            rows.append(d)
            return True
        return False

    # A. Contraste Clima (14 instancias / 7 pares)
    pares_c = 1
    # Mezclamos las combinaciones posibles para no sesgar a los primeros días
    combinaciones_clima = [(d, h) for d in [0, 2, 4, 5] for h in [0, 6, 12, 18]]
    rng.shuffle(combinaciones_clima)

    for d_sem, hora in combinaciones_clima:
        if pares_c > 7: break
        
        # Filtramos quitando lo ya seleccionado ANTES de elegir
        pool = df[(df['DIA_SEM'] == d_sem) & (df['HORA'] == hora) & (~df['FECHA_FRANJA'].isin(seleccionados))]
        
        lluvia_pool = pool[pool['PRECIP_SUM_PREV'] > 0.5]
        seco_pool = pool[pool['PRECIP_SUM_PREV'] == 0]
        
        if not lluvia_pool.empty and not seco_pool.empty:
            lluvia = lluvia_pool.iloc[rng.choice(len(lluvia_pool))]
            seco = seco_pool.iloc[rng.choice(len(seco_pool))]
            
            add_r(lluvia, 'CLIMA_CONTRASTE', f"CLIMA_{str(pares_c).zfill(2)}_LLUVIA")
            add_r(seco, 'CLIMA_CONTRASTE', f"CLIMA_{str(pares_c).zfill(2)}_SECO")
            pares_c += 1

    # B. Contraste Rutina (14 instancias / 7 pares) - CORREGIDO EL BUCLE
    pares_r = 1
    # Ampliamos a todas las horas para garantizar conseguir los 7 pares
    horas_rutina = [0, 6, 12, 18] * 3  # Repetimos la lista para permitir múltiples pares por hora
    rng.shuffle(horas_rutina)

    for hora in horas_rutina:
        if pares_r > 7: break
        
        pool_r = df[(df['HORA'] == hora) & (df['PRECIP_SUM_PREV'] == 0) & (~df['FECHA_FRANJA'].isin(seleccionados))]
        habiles = pool_r[pool_r['FNSM'] == 0]
        findes = pool_r[pool_r['FNSM'] == 1]
        
        if not habiles.empty and not findes.empty:
            r_habil = habiles.iloc[rng.choice(len(habiles))]
            r_finde = findes.iloc[rng.choice(len(findes))]
            
            add_r(r_habil, 'RUTINA_CONTRASTE', f"RUTINA_{str(pares_r).zfill(2)}_HABIL")
            add_r(r_finde, 'RUTINA_CONTRASTE', f"RUTINA_{str(pares_r).zfill(2)}_FINDE")
            pares_r += 1

    # C. Evolución 24h (32 instancias / 8 días)
    dias_pool = [d for d in df['FECHA'].unique() if d not in {f.date() for f in seleccionados}]
    rng.shuffle(dias_pool)
    d_count = 1
    for d in dias_pool:
        if d_count > 8: break
        bloque = df[df['FECHA'] == d].sort_values('FECHA_FRANJA')
        if len(bloque) == 4 and not (set(bloque['FECHA_FRANJA']) & seleccionados):
            for _, r in bloque.iterrows(): 
                h_str = str(r['HORA']).zfill(2)
                add_r(r, 'EVOLUCION_24H', f"EVO_{str(d_count).zfill(2)}_{h_str}H")
            d_count += 1

    # CONSOLIDACIÓN
    df_res = pd.DataFrame(rows)
    
    dias_map = {0:'Lunes', 1:'Martes', 2:'Miércoles', 3:'Jueves', 4:'Viernes', 5:'Sábado', 6:'Domingo'}
    meses_map = {1:'Enero', 2:'Febrero', 3:'Marzo', 4:'Abril', 5:'Mayo', 6:'Junio', 
                 7:'Julio', 8:'Agosto', 9:'Septiembre', 10:'Octubre', 11:'Noviembre', 12:'Diciembre'}
    franja_map = {0:'0:00 - 6:00', 6:'6:00 - 12:00', 12:'12:00 - 18:00', 18:'18:00 - 0:00'}

    df_res['DIA'] = df_res['DIA_SEM'].map(dias_map)
    df_res['MES'] = df_res['FECHA_FRANJA'].dt.month.map(meses_map)
    df_res['FRANJA'] = df_res['HORA'].map(franja_map)

    # Agregar conteo de accidentes reales
    conteo_acc = df_accidentes_test.groupby('FECHA_FRANJA').size().reset_index(name='TOTAL_ACCIDENTES')
    df_res = pd.merge(df_res, conteo_acc, on='FECHA_FRANJA', how='left')
    df_res['TOTAL_ACCIDENTES'] = df_res['TOTAL_ACCIDENTES'].fillna(0).astype(int)

    # ordenamiento y selección final de columnas
    cols_contexto = ['ID_FRANJA', 'FECHA_FRANJA', 'TOTAL_ACCIDENTES', 'METODOLOGIA', 'MES', 'DIA', 'FRANJA']
    cols_cal = [col for col in df_calendario.columns if col not in ['FECHA']]
    cols_prev = [f'{c}_PREV' for c in cols_clima_base if f'{c}_PREV' in df_res.columns]
    
    # Ordenamos por ID_FRANJA para que los gemelos aparezcan juntos automáticamente
    df_final = df_res[cols_contexto + cols_cal + cols_prev].sort_values(['METODOLOGIA', 'ID_FRANJA']).reset_index(drop=True)
    
    return df_final

# Uso:
df_instancias_test = generar_dataset_test(df_cal, df_meteo, df_acc_test)

print("Total de instancias:", len(df_instancias_test))
display(df_instancias_test.head(10))

Total de instancias: 60


,ID_FRANJA,FECHA_FRANJA,TOTAL_ACCIDENTES,METODOLOGIA,MES,DIA,FRANJA,FESTIVO,FNSM,VACACIONES,PROMOCION,PANDEMIA,WIND_SPEED_MEAN_PREV,SOIL_MOISTURE_MEAN_PREV,APP_TEMP_MEAN_PREV,TEMP_MEAN_PREV,WIND_GUSTS_MAX_PREV,SOIL_TEMP_MEAN_PREV,TEMP_MIN_PREV,CLOUD_COVER_MEAN_PREV,PRECIP_SUM_PREV,PRECIP_MAX_PREV
0,CLIMA_01_LLUVIA,2024-07-20 18:00:00,1,CLIMA_CONTRASTE,Julio,Sábado,18:00 - 0:00,0,0,1,0,0,6.223333,0.378000,23.970049,22.449999,26.639999,25.074999,21.049999,80.666664,1.2,0.9
1,CLIMA_01_SECO,2024-04-20 18:00:00,5,CLIMA_CONTRASTE,Abril,Sábado,18:00 - 0:00,0,1,0,0,0,20.610901,0.119000,26.230268,30.633333,49.320000,35.875000,28.850000,21.833334,0.0,0.0
2,CLIMA_02_LLUVIA,2024-06-07 18:00:00,3,CLIMA_CONTRASTE,Junio,Viernes,18:00 - 0:00,0,0,0,0,0,8.302051,0.122333,28.905115,29.824999,45.360001,39.600002,27.700001,50.000000,0.6,0.5
3,CLIMA_02_SECO,2024-12-13 18:00:00,3,CLIMA_CONTRASTE,Diciembre,Viernes,18:00 - 0:00,0,0,0,0,0,6.520466,0.146500,21.412683,23.525335,22.320000,25.033333,21.944000,34.166668,0.0,0.0
4,CLIMA_03_LLUVIA,2024-07-26 12:00:00,1,CLIMA_CONTRASTE,Julio,Viernes,12:00 - 18:00,0,0,1,0,0,5.356384,0.428500,16.962034,16.158333,17.280001,17.525000,15.300000,100.000000,5.9,2.9
5,CLIMA_03_SECO,2024-08-23 12:00:00,1,CLIMA_CONTRASTE,Agosto,Viernes,12:00 - 18:00,0,0,1,0,0,6.870167,0.212667,17.525034,17.275000,21.600000,18.941668,13.450000,25.500000,0.0,0.0
6,CLIMA_04_LLUVIA,2024-07-12 06:00:00,0,CLIMA_CONTRASTE,Julio,Viernes,6:00 - 12:00,0,0,0,0,0,5.864383,0.339833,15.970401,15.407666,15.120000,17.240999,14.850000,95.500000,0.6,0.2
7,CLIMA_04_SECO,2024-11-08 06:00:00,2,CLIMA_CONTRASTE,Noviembre,Viernes,6:00 - 12:00,0,0,0,0,0,4.691383,0.136000,13.656250,14.491666,12.240000,16.591667,13.039500,55.166668,0.0,0.0
8,CLIMA_05_LLUVIA,2024-07-08 18:00:00,1,CLIMA_CONTRASTE,Julio,Lunes,18:00 - 0:00,0,0,0,0,0,6.192450,0.267167,25.941534,23.908333,33.480000,27.975000,19.750000,76.500000,6.9,5.3
9,CLIMA_05_SECO,2024-04-29 18:00:00,2,CLIMA_CONTRASTE,Abril,Lunes,18:00 - 0:00,0,1,0,0,0,11.709816,0.119000,25.548082,28.683332,32.759998,35.450001,26.650000,76.833336,0.0,0.0


## 7. Funciones de integración

Se busca para cada franja con accidente la arista más cercana dentro de la red, de esta manera, cada franja queda asociada a un tramo vial específico. Posteriormente, cada franja del conjunto de datos de los accidentes se relaciona de manera directa con la fecha del calendario de eventos.

Finalmente, la variable FECHA_FRANJA se transforma en variables temporales cíclicas, aplicando funciones seno y coseno para representar estos periodos de manera continua (mes, día de la semana y las franjas horarias).

In [ ]:
# GENERADOR DE REGISTROS
def generador_registros(fecha: pd.DatetimeIndex, gdf_red_vial: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Genera un DataFrame de registros base para una fecha/franja horaria específica.

    Parámetros
    ----------
    fecha: Fecha y franja horaria (ej. 2024-01-01 06:00:00).

    gdf_red_vial: DataFrame con la red vial, debe contener la columna 'ID_RED'.

    Retorna
    -------
    df_integrado: DataFrame con 
        columnas:
        - 'ID_RED' : identificador de tramo vial.
        - 'FECHA_FRANJA' : fecha/franja asignada.
        - 'DIASEMANA' : día de la semana (0=lunes, 6=domingo).
    """
    df_integrado = gdf_red_vial[['ID_RED']].copy()
    df_integrado["FECHA_FRANJA"] = fecha
    df_integrado["DIASEMANA"] = fecha.weekday()
    return df_integrado

# INTEGRACIÓN REGISTROS - FECHAS
def registros_fechas(df_registros: pd.DataFrame, df_calendario: pd.DataFrame, muestra = "entrenamiento"):
    """
    Integra registros de red vial con calendario de fechas, adaptando el cruce según el tipo de muestra (entrenamiento o prueba).

    Parámetros
    ----------
    df_registros: DataFrame con 
        columnas 'ID_RED' y 'FECHA_FRANJA'.

    df_calendario: DataFrame calendario 
        con columnas:
        - 'FECHA' : fecha normalizada (datetime).
        - 'FESTIVO', 'FNSM', 'VACACIONES', etc.

    muestra : Define el modo de integración 
        (default="entrenamiento"):
        - "entrenamiento": se hace un merge clásico por fecha.
        - "prueba": se asignan los valores de calendario de la única fecha compartida.

    Retorna
    -------
    df_integrado : DataFrame con registros integrados con variables de calendario.
    """
    # Copias de seguridad
    temp_registros = df_registros.copy()
    temp_calendario = df_calendario.copy()

    # Columnas del calendario
    columnas_cal = temp_calendario.columns.to_list()
    columnas_cal.remove('FECHA')

    # Si son registros de prueba, comparten a misma franja
    if muestra == "prueba":
        if not temp_registros.empty:
            # Al compartir la misma fecha, extraemos la primera como referencia
            fecha_unica = temp_registros['FECHA_FRANJA'].iloc[0].normalize()
            
            # Buscamos esa única fecha en el calendario
            coincidencia = df_calendario[pd.to_datetime(df_calendario['FECHA']).dt.normalize() == fecha_unica]
            
            if not coincidencia.empty:
                valores = coincidencia[columnas_cal].iloc[0].tolist()
            else:
                valores = [0 for _ in range(len(columnas_cal))]
        else:
            valores = [0 for _ in range(len(columnas_cal))]
            
        # Propagación de valores
        temp_registros[columnas_cal] = valores
        df_integrado = temp_registros

    # Si son registros de entrenamiento, será empatar las fechas
    else:
        # Normalizamos las fechas en ambos dataframes para hacer un cruce limpio
        temp_registros['FECHA'] = temp_registros['FECHA_FRANJA'].dt.normalize()
        temp_calendario['FECHA'] = pd.to_datetime(temp_calendario['FECHA']).dt.normalize()

        # Cruce Left Join
        df_integrado = pd.merge(temp_registros, temp_calendario, on='FECHA', how='left')
        df_integrado.drop(columns=['FECHA'], inplace=True)

    # Rellenamos nulos con 0 y bajamos el peso en memoria a int8 para todo el bloque de columnas
    df_integrado[columnas_cal] = df_integrado[columnas_cal].fillna(0).astype('int8')

    return df_integrado

# INTEGRACIÓN REGISTROS - RED VIAL 
def registros_red(df_registros: pd.DataFrame, gdf_red_vial: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Integra registros con atributos de la red vial, excluyendo la geometría.

    Parámetros
    ----------
    df_registros : DataFrame con 
        columna 'ID_RED'.

    gdf_red_vial : GeoDataFrame con información de la red vial, debe contener 'ID_RED'
        y atributos adicionales (ej. tipo de vialidad, carriles, velocidad).

    Retorna
    -------
    df_integrado : DataFrame resultante del merge, con atributos de la red vial asociados a cada registro.
    """
    # No tomar en cuenta la geometría
    df_red_sin_geo = pd.DataFrame(gdf_red_vial.drop(columns=[gdf_red_vial.geometry.name, 'UNION_INI', 'UNION_FIN'], errors='ignore'))
    
    # Merge
    df_integrado = df_registros.merge(
        df_red_sin_geo,
        on="ID_RED",
        how="left"
    )
    
    return df_integrado

# CONVERTIR FECHAS EN CICLOS
def transformar_ciclicas(df_registros: pd.DataFrame) -> pd.DataFrame:
    """
    Convierte variables temporales en representaciones cíclicas mediante transformaciones seno y coseno,
    para preservar la naturaleza periódica del tiempo en modelos de Machine Learning.

    Parámetros
    ----------
    df_registros: DataFrame con columna 
        'FECHA_FRANJA' (datetime) y 'DIASEMANA'.

    Retorna
    -------
    df_transf: DataFrame con 
        nuevas columnas:
        - 'MES_SIN', 'MES_COS' : codificación cíclica del mes (1-12).
        - 'DIASEMANA_SIN', 'DIASEMANA_COS' : codificación cíclica del día de semana (0-6).
        - 'FRANJA_SIN', 'FRANJA_COS' : codificación cíclica de la franja horaria (0-23).
    """
    df_transf = df_registros.copy()
    
    # Extraer variables
    df_transf['MES'] = df_transf['FECHA_FRANJA'].dt.month
    df_transf['HORA_FRANJA'] = df_transf['FECHA_FRANJA'].dt.hour
    
    # Parámetros máximos para cada ciclo
    MESES_MAX = 12
    DIAS_MAX = 7
    HORAS_MAX = 24
    
    # Transformación de MES
    df_transf['MES_SIN'] = np.round(np.sin(2 * np.pi * df_transf['MES'] / MESES_MAX), 6).astype('float32')
    df_transf['MES_COS'] = np.round(np.cos(2 * np.pi * df_transf['MES'] / MESES_MAX), 6).astype('float32')
    
    # Transformación de DÍA DE SEMANA
    df_transf['DIASEMANA_SIN'] = np.round(np.sin(2 * np.pi * df_transf['DIASEMANA'] / DIAS_MAX), 6).astype('float32')
    df_transf['DIASEMANA_COS'] = np.round(np.cos(2 * np.pi * df_transf['DIASEMANA'] / DIAS_MAX), 6).astype('float32')
    
    # Transformación de FRANJA HORARIA
    df_transf['FRANJA_SIN'] = np.round(np.sin(2 * np.pi * df_transf['HORA_FRANJA'] / HORAS_MAX), 6).astype('float32')
    df_transf['FRANJA_COS'] = np.round(np.cos(2 * np.pi * df_transf['HORA_FRANJA'] / HORAS_MAX), 6).astype('float32')
    
    # Eliminación de las variables originales temporales
    columnas_a_borrar = ['MES', 'DIASEMANA', 'HORA_FRANJA']
    df_transf = df_transf.drop(columns=[c for c in columnas_a_borrar if c in df_transf.columns])
    
    return df_transf

## 8. Funciones de variables de memoria


Se unieron los registros con su franja meteorológica anterior por el hecho de que, en escenarios prácticos de predicción, la información disponible en la franja del modelado siempre corresponde a la información pasada.

Las variables de memoria temporal se calcularon contabilizando la ocurrencia de accidentes de manera local. A cada registro de la muestra se contabilizan los accidentes ocurridos en esa arista en intervalos previos de tiempo: dos semanas, y uno y dos meses.

In [12]:
# INTEGRACION CON CLIMA PREVIO
def registro_clima_anterior(df_registros: pd.DataFrame, df_meteo: pd.DataFrame) -> pd.DataFrame:
    """
    Integra registros con información meteorológica de la franja anterior.

    Parámetros
    ----------
    df_registros: DataFrame con registros base, 
        debe contener:
        - 'ID_CENTRO' : identificador del centro meteorológico.
        - 'FECHA_FRANJA' : fecha y franja horaria (datetime).
    meteo: DataFrame con datos meteorológicos 
        por centro y franja.

    Retorna
    -------
    df_integrado : DataFrame con registros integrados con variables climáticas de la franja anterior.
        Las columnas se renombran con sufijo '_PREV' para indicar que corresponden a la franja previa. 
        """
    # Copia de seguridad
    df_clima = df_meteo.copy()
    # El clima real que ocurrió de 00:00 a 05:59, ahora se etiquetará como "06:00".
    df_clima['FECHA_FRANJA'] = df_clima['FECHA_FRANJA'] + pd.Timedelta(hours=6)
        
    # Renombrar todas las columnas excepto las claves
    columnas_clima = [c for c in df_clima.columns if c not in ['ID_CENTRO','FECHA_FRANJA']]
    rename_dict = {col: f"{col}_PREV" for col in columnas_clima}
    df_clima.rename(columns=rename_dict, inplace=True)
    
    # Merge relacionando franjas anteriores
    df_integrado = df_registros.merge(
        df_clima,
        on=["ID_CENTRO", "FECHA_FRANJA"],
        how="left"
    )

    # Rellenar nulos (por si algún centro meteorológico se apagó en alguna franja)
    df_integrado = df_integrado.sort_values(['ID_CENTRO', 'FECHA_FRANJA'])
    columnas_clima_pasado = [c for c in df_integrado.columns if c.endswith('_PREV')]
    df_integrado[columnas_clima_pasado] = (
        df_integrado.groupby('ID_CENTRO')[columnas_clima_pasado].ffill()
    )
    df_integrado.drop(columns=['ID_CENTRO'], inplace = True)
    
    # Reordenar según el orden original de registros
    df_integrado = df_integrado.loc[df_registros.index]

    return df_integrado

# INTEGRACION VARIABLE DE MEMORIA LOCAL DE ACCIDENTES
def calcular_memoria_accidentes(df_muestra: pd.DataFrame, df_accidentes: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula variables de memoria de accidentes locales en ventanas temporales.

    Parámetros
    ----------
    df_muestra: DataFrame de muestra 
        con columnas:
        - 'ID_RED' : identificador de tramo vial.
        - 'FECHA_FRANJA' : fecha y franja horaria (datetime).
    df_accidentes: Histórico de accidentes 
        con columnas:
        - 'ID_RED'
        - 'FECHA_FRANJA'

    Retorna
    -------
    df_muestra_lags: DataFrame con 
        columnas adicionales:
        - 'ACC_LOCAL_2W' : número de accidentes en las últimas 2 semanas.
        - 'ACC_LOCAL_1M' : número de accidentes entre 2 y 4 semanas atrás.
        - 'ACC_LOCAL_2M' : número de accidentes entre 4 y 8 semanas atrás.
    """
    # Histórico de accidentes como un índice para búsqueda
    df_hist_idx = df_accidentes[['ID_RED', 'FECHA_FRANJA']].copy()
    df_hist_idx = df_hist_idx.set_index('ID_RED').sort_values('FECHA_FRANJA')
    
    def calcular_fila(row):
        fecha_actual = row['FECHA_FRANJA']
        id_red_actual = row['ID_RED']
        
        # Ventanas temporales exactas (múltiplos de 7 días)
        fecha_2w = fecha_actual - pd.Timedelta(days=14)
        fecha_1m = fecha_actual - pd.Timedelta(days=28)
        fecha_2m = fecha_actual - pd.Timedelta(days=56)
        
        try:
            locales_data = df_hist_idx.loc[[id_red_actual]]
            # Filtramos estrictamente anterior a la franja actual
            locales_data = locales_data[locales_data['FECHA_FRANJA'] < fecha_actual]
            
            # Contabilizamos cuántas franjas tuvieron accidente en cada bloque de tiempo
            # Bloque 1: De hace 2 semanas hasta hoy
            acc_local_2w = (locales_data['FECHA_FRANJA'] >= fecha_2w).sum()
            # Bloque 2: De hace 4 semanas (1 mes) hasta hace 2 semanas
            acc_local_1m = ((locales_data['FECHA_FRANJA'] >= fecha_1m) & 
                            (locales_data['FECHA_FRANJA'] < fecha_2w)).sum()
            # Bloque 3: De hace 8 semanas (2 meses) hasta hace 4 semanas
            acc_local_2m = ((locales_data['FECHA_FRANJA'] >= fecha_2m) & 
                            (locales_data['FECHA_FRANJA'] < fecha_1m)).sum()
        except KeyError:
            # Si el ID_RED no existe en el histórico
            acc_local_2w = acc_local_1m = acc_local_2m = 0
        return pd.Series([
            acc_local_2w, acc_local_1m, acc_local_2m
        ])

    columnas_nuevas = ['ACC_LOCAL_2W', 'ACC_LOCAL_1M', 'ACC_LOCAL_2M']
    df_muestra_lags = df_muestra.copy()

    # Uso de SWIFTER para paralelizar el apply
    df_muestra_lags[columnas_nuevas] = df_muestra_lags.swifter.apply(calcular_fila, axis=1)

    # Convertir todas las nuevas columnas a int16
    df_muestra_lags[columnas_nuevas] = df_muestra_lags[columnas_nuevas].astype("int16")
    
    return df_muestra_lags

## 9. Integración de conjunto de entrenamiento

In [13]:
ruta_entrenamiento = "Generados/df_entrenamiento.parquet"
if not os.path.exists(ruta_entrenamiento):
    # Aplicando funciones de integración y transformación
    print('Integrando registros con fechas')
    df_train_integrado = registros_fechas(df_train, df_cal)
    print('Integrando registros con red vial')
    df_train_integrado = registros_red(df_train_integrado, gdf_rnc)
    print('Transformando variables cíclicas')
    df_train_integrado = transformar_ciclicas(df_train_integrado)
    print('Integrando registros con clima anterior')
    df_train_integrado = registro_clima_anterior(df_train_integrado, df_meteo)
    print('Calculando memoria local de accidentes')
    df_train_integrado = calcular_memoria_accidentes(df_train_integrado, df_acc)
    print('Proceso de integración completo. Dataframe listo para modelar.')
    # Guardado del conjunto de entrenamiento
    df_train_integrado.to_parquet(ruta_entrenamiento, index=False)
    print(f"Conjunto de entrenamiento guardada: {ruta_entrenamiento}")
else:
    print(f"El archivo {ruta_entrenamiento} ya existe. Cargando...")
    df_train_integrado = pd.read_parquet(ruta_entrenamiento)
print("Información del conjunto de entrenamiento integrado:")
display(df_train_integrado.head(5))
print(df_train_integrado.info())

El archivo Generados/df_entrenamiento.parquet ya existe. Cargando...
Información del conjunto de entrenamiento integrado:


,ID_RED,FECHA_FRANJA,TARGET,TIPO_MUESTRA,FESTIVO,FNSM,VACACIONES,PROMOCION,PANDEMIA,CARRILES,NIVEL,VELOCIDAD,LONGITUD,AVENIDA,BOULEVARD,CALLE,CAMINO,CARRETERA,CIRCUITO,GLORIETA,INCORPORACION,PRIVADA,RETORNO_U,MENOR,CERRADA_EN_AMBOS_SENTIDOS,DOS_SENTIDOS,UN_SENTIDO,GRADO_MIN,GRADO_MAX,DIST_NODO_3,DIST_NODO_4,DIST_NODO_5_MAS,SINUOSIDAD,DIST_MAN_PROH,DELTA_CARRILES_MAX,DELTA_CARRILES_MIN,DELTA_CARRILES_MEAN,DELTA_VELOCIDAD_MAX,DELTA_VELOCIDAD_MIN,DELTA_VELOCIDAD_MEAN,MES_SIN,MES_COS,DIASEMANA_SIN,DIASEMANA_COS,FRANJA_SIN,FRANJA_COS,PRECIP_SUM_PREV,PRECIP_MAX_PREV,WIND_GUSTS_MAX_PREV,WIND_SPEED_MEAN_PREV,TEMP_MEAN_PREV,TEMP_MIN_PREV,APP_TEMP_MEAN_PREV,CLOUD_COVER_MEAN_PREV,SOIL_TEMP_MEAN_PREV,SOIL_MOISTURE_MEAN_PREV,ACC_LOCAL_2W,ACC_LOCAL_1M,ACC_LOCAL_2M
0,877898_2,2019-02-28 06:00:00,1,accidente_real,0,0,0,0,0,2,1,50,141.243103,1,0,0,0,0,0,0,0,0,0,0,0,0,1,2,2,17.474588,18.400482,1261.830322,1.000215,771.981873,0,0,0.00,0,0,0.0,0.866025,0.5,0.433884,-0.900969,1.0,0.0,0.0,0.0,11.520000,6.623966,14.508334,13.500000,11.587700,60.666668,16.266666,0.088000,0,0,1
1,550482,2019-02-28 06:00:00,1,accidente_real,0,0,0,0,0,4,0,50,66.566269,1,0,0,0,0,0,0,0,0,0,0,0,0,1,3,3,33.468681,44.851730,830.587769,1.000029,215.829987,2,0,1.00,20,0,10.0,0.866025,0.5,0.433884,-0.900969,1.0,0.0,0.0,0.0,11.520000,6.623966,14.508334,13.500000,11.587700,60.666668,16.266666,0.088000,0,0,0
2,550075_1,2019-02-28 00:00:00,1,accidente_real,0,0,0,0,0,2,-1,50,135.857803,1,0,0,0,0,0,0,0,0,0,0,0,0,1,2,2,7.733476,34.290611,437.436035,1.000253,82.265244,0,0,0.00,0,0,0.0,0.866025,0.5,0.433884,-0.900969,0.0,1.0,0.0,0.0,24.480000,10.657967,20.616667,16.450001,16.645000,0.500000,25.308334,0.143167,0,0,0
3,4838974,2019-02-27 06:00:00,1,accidente_real,0,0,0,0,0,2,0,50,27.006979,0,1,0,0,0,0,0,0,0,0,0,0,0,1,3,3,13.579928,22.655237,948.448669,1.000000,41.158081,1,0,0.25,20,0,5.0,0.866025,0.5,0.974928,-0.222521,1.0,0.0,0.0,0.0,17.639999,7.038633,13.050000,10.800000,9.902467,0.000000,16.333334,0.139000,0,0,0
4,2639756,2019-02-28 00:00:00,1,accidente_real,0,0,0,0,0,4,0,50,127.648033,1,0,0,0,0,0,0,0,0,0,0,0,0,1,3,4,58.834480,64.181763,223.374161,1.000028,142.513748,3,0,0.80,20,0,8.0,0.866025,0.5,0.433884,-0.900969,0.0,1.0,0.0,0.0,25.200001,9.956166,21.533333,17.000000,17.786451,5.500000,25.000000,0.088000,0,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76722 entries, 0 to 76721
Data columns (total 59 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   ID_RED                     76722 non-null  object        
 1   FECHA_FRANJA               76722 non-null  datetime64[ns]
 2   TARGET                     76722 non-null  int8          
 3   TIPO_MUESTRA               76722 non-null  object        
 4   FESTIVO                    76722 non-null  int8          
 5   FNSM                       76722 non-null  int8          
 6   VACACIONES                 76722 non-null  int8          
 7   PROMOCION                  76722 non-null  int8          
 8   PANDEMIA                   76722 non-null  int8          
 9   CARRILES                   76722 non-null  int8          
 10  NIVEL                      76722 non-null  int8          
 11  VELOCIDAD                  76722 non-null  int16         
 12  LONG

## 10. Integración de conjuntos de prueba

In [14]:
# Configuración de la carpeta única
ruta_carpeta = 'Generados/Instancias_Prueba'
if not os.path.exists(ruta_carpeta):
    os.makedirs(ruta_carpeta)

# Nombramiento
if 'ID_FRANJA' not in df_instancias_test.columns:
    df_instancias_test['ID_FRANJA'] = [f"INST_{str(i+1).zfill(2)}" for i in range(len(df_instancias_test))]

# Columna ID_INSTANCIA con el formato Instancia_1, Instancia_2, ...
df_instancias_test['ID_INSTANCIA'] = [f"Instancia_{i+1}" for i in range(len(df_instancias_test))]
df_instancias_test = df_instancias_test[['ID_INSTANCIA']+[col for col in df_instancias_test.columns if col != 'ID_INSTANCIA']]
# Guardar el archivo de relación (crucial para saber qué ID corresponde a qué contexto)
df_instancias_test.to_csv(f'{ruta_carpeta}/relacion_instancias.csv', index=False)

print(f"Iniciando procesamiento de {len(df_instancias_test)} instancias de prueba")
print(f"Carpeta de destino: {ruta_carpeta}")

for idx, fila in df_instancias_test.iterrows():
    fecha = fila['FECHA_FRANJA']
    id_franja = fila['ID_FRANJA']
    id_instancia = fila['ID_INSTANCIA']
    metodologia = fila['METODOLOGIA'] # Solo para que el print sea más informativo
    
    # Control de existencia
    nombre_archivo = f"{id_instancia}.parquet"
    ruta_completa_archivo = os.path.join(ruta_carpeta, nombre_archivo)
    
    if os.path.exists(ruta_completa_archivo):
        print(f"[-] Omitiendo: {id_instancia} ({fecha}) - Ya existe")
        continue 

    print(f"[+] Procesando: {id_instancia} | Fecha: {fecha} | {metodologia}")
    
    try:
        # Integración
        df_inst = generador_registros(fecha, gdf_rnc)
        df_inst = registros_fechas(df_inst, df_cal)
        df_inst = registros_red(df_inst, gdf_rnc)
        df_inst = transformar_ciclicas(df_inst)
        df_inst = registro_clima_anterior(df_inst, df_meteo)
        df_inst = calcular_memoria_accidentes(df_inst, df_acc)
        
        # Guardado individual
        df_inst.to_parquet(ruta_completa_archivo, index=False)
        
    except Exception as e:
        print(f"[!] Error crítico procesando {id_instancia} ({fecha}): {e}")
        continue
    
print("Integración de instancias de prueba finalizado")

Iniciando procesamiento de 60 instancias de prueba
Carpeta de destino: Generados/Instancias_Prueba
[-] Omitiendo: Instancia_1 (2024-07-20 18:00:00) - Ya existe
[-] Omitiendo: Instancia_2 (2024-04-20 18:00:00) - Ya existe
[-] Omitiendo: Instancia_3 (2024-06-07 18:00:00) - Ya existe
[-] Omitiendo: Instancia_4 (2024-12-13 18:00:00) - Ya existe
[-] Omitiendo: Instancia_5 (2024-07-26 12:00:00) - Ya existe
[-] Omitiendo: Instancia_6 (2024-08-23 12:00:00) - Ya existe
[-] Omitiendo: Instancia_7 (2024-07-12 06:00:00) - Ya existe
[-] Omitiendo: Instancia_8 (2024-11-08 06:00:00) - Ya existe
[-] Omitiendo: Instancia_9 (2024-07-08 18:00:00) - Ya existe
[-] Omitiendo: Instancia_10 (2024-04-29 18:00:00) - Ya existe
[-] Omitiendo: Instancia_11 (2024-07-01 00:00:00) - Ya existe
[-] Omitiendo: Instancia_12 (2024-03-11 00:00:00) - Ya existe
[-] Omitiendo: Instancia_13 (2024-02-14 18:00:00) - Ya existe
[-] Omitiendo: Instancia_14 (2024-06-05 18:00:00) - Ya existe
[-] Omitiendo: Instancia_15 (2024-10-08 00